# AWREAS Direct Scoring Evaluator
**Metrics:** Quadratic Weighted Kappa (QWK) · Mean Absolute Error (MAE) · Spearman Correlation

This notebook benchmarks AWREAS scoring performance directly inside Colab — **no running API server needed**.
It calls the supervisor pipeline in-process, collects predicted scores, rescales them to the ASAP Prompt 1
range (2–12), and computes agreement with human gold scores.

**Pipeline per essay:**
1. Load essay from `data/asap/benchmark_prompt1.jsonl`
2. Run all 8 worker agents + Critic + Synthesis via `supervisor.run_evaluation()`
3. Collect `context.overall_score` (0–100 scale)
4. Rescale to 2–12 and compare with gold

**Before running:** Upload `academic-review-system.zip` to Google Drive.

## Cell 1 — Mount Drive and Extract Project

In [ ]:
from google.colab import drive
import os, sys, glob

drive.mount('/content/drive')

zip_path    = '/content/drive/MyDrive/academic-review-system.zip'
extract_dir = '/content/academic-review-system'

if os.path.exists(zip_path):
    print('Found zip — extracting...')
    !unzip -q -o "{zip_path}" -d "{extract_dir}"
    print('Extraction complete.')
else:
    extract_dir = '/content/drive/MyDrive/academic-review-system'
    if not os.path.exists(extract_dir):
        raise FileNotFoundError('Upload academic-review-system.zip to Google Drive first.')

toml_paths = glob.glob(os.path.join(extract_dir, '**/pyproject.toml'), recursive=True)
if not toml_paths:
    raise FileNotFoundError('Cannot locate pyproject.toml inside extracted files.')

project_root = os.path.dirname(toml_paths[0])
print(f'Project root: {project_root}')
sys.path.insert(0, project_root)
%cd "{project_root}"

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q .
!pip install -q google-genai
print('Dependencies installed.')

## Cell 3 — Configure LLM Provider and API Key

In [ ]:
import getpass

print('Available providers: gemini | openrouter | openai | qwen')
provider = input('Enter LLM provider: ').strip().lower()
if provider not in ('gemini', 'openrouter', 'openai', 'qwen'):
    print("Unrecognised — defaulting to 'openai'.")
    provider = 'openai'

api_key = getpass.getpass(f'Paste your {provider.upper()} API key: ').strip()
if not api_key:
    raise ValueError('API key is required.')

MODEL_DEFAULTS = {
    'openai':     'gpt-4o-mini',
    'openrouter': 'anthropic/claude-sonnet-4.6',
    'gemini':     'gemini-2.5-flash',
    'qwen':       'qwen3.7-max',
}
default_model = MODEL_DEFAULTS[provider]
model_input   = input(f'Model name [{default_model}]: ').strip()
model_name    = model_input if model_input else default_model

print(f'\nProvider : {provider}')
print(f'Model    : {model_name}')
print('Ready.')

## Cell 4 — Initialise AWREAS

In [ ]:
# Clear any stale cached imports from a previous run in this session
for mod in [k for k in sys.modules if k.startswith('app')]:
    del sys.modules[mod]

from app.core.config import settings
settings.ENABLE_WEB_RESEARCH = False  # prevents slow external lookups during benchmarking
settings.LLM_REQUEST_TIMEOUT = 60.0

settings.WORKER_MODEL_NAME    = model_name
settings.SYNTHESIS_MODEL_NAME = model_name
settings.REVIEW_MODEL_NAME    = model_name
settings.REVISION_MODEL_NAME  = model_name

from app.llm.client import LLMClient
from app.supervisor.agent import SupervisorAgent

llm_client = LLMClient(api_key=api_key, provider=provider, default_model=model_name)
supervisor  = SupervisorAgent(llm_client=llm_client)

print(f'SupervisorAgent ready (provider={provider}, model={model_name}).')
print(f'LLM available: {llm_client.available}')

## Cell 5 — Load ASAP Prompt 1 Dataset and Gold Scores

In [ ]:
import json
from pathlib import Path

BENCHMARK_FILE = Path('data/asap/benchmark_prompt1.jsonl')
GOLD_FILE      = Path('data/asap/gold_prompt1.json')

if not BENCHMARK_FILE.exists():
    print('benchmark_prompt1.jsonl not found — running prepare script...')
    !python scripts/prepare_asap_benchmark.py

with open(BENCHMARK_FILE, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f if line.strip()]

with open(GOLD_FILE, 'r', encoding='utf-8') as f:
    gold_scores = json.load(f)  # {"1": 8.0, "2": 9.0, ...}

print(f'Dataset   : {len(dataset)} essays loaded from benchmark_prompt1.jsonl')
print(f'Gold file : {len(gold_scores)} scores loaded from gold_prompt1.json')
print(f'Gold range: {min(gold_scores.values()):.0f} – {max(gold_scores.values()):.0f} '
      f'(ASAP Prompt 1 scale: 2–12)')

## Cell 6 — Sampling and Run Configuration

Each essay is evaluated `RUNS_PER_ESSAY` times. The mean score across runs is used
as the final prediction — this controls for LLM stochasticity, consistent with the
repeated-run protocol described in Chapter 3.

In [ ]:
# ---------------------------------------------------------------
# CONFIGURATION — adjust as needed
# ---------------------------------------------------------------
SAMPLE_SIZE    = 30    # Number of essays to evaluate. Set to None for all 145.
RUNS_PER_ESSAY = 3     # Repeated runs per essay for score stability measurement.
GOLD_MIN       = 2     # ASAP Prompt 1 minimum score
GOLD_MAX       = 12    # ASAP Prompt 1 maximum score
# ---------------------------------------------------------------

essays_to_run = dataset[:SAMPLE_SIZE] if SAMPLE_SIZE else dataset

# Keep only essays that have a gold score
essays_to_run = [e for e in essays_to_run if str(e['id']) in gold_scores]

print(f'Essays to evaluate : {len(essays_to_run)}')
print(f'Runs per essay     : {RUNS_PER_ESSAY}')
print(f'Total LLM calls    : ~{len(essays_to_run) * RUNS_PER_ESSAY * 8} '
      f'(essays × runs × 8 workers)')

## Cell 7 — Run Evaluation

Each essay is submitted to the full 8-agent supervisor pipeline `RUNS_PER_ESSAY` times.
Worker scores, overall score, Critic approval, and wall latency are recorded for each run.

In [ ]:
import time, statistics as st

# records[essay_id] = list of per-run dicts
records = {}   # {essay_id: [{overall_score, worker_scores, critic_approved, wall_ms}, ...]}
failed  = []   # list of (essay_id, run, error_msg)

total_essays = len(essays_to_run)
print(f'Starting evaluation: {total_essays} essays × {RUNS_PER_ESSAY} runs each...\n')

for i, essay in enumerate(essays_to_run):
    essay_id  = str(essay['id'])
    essay_text = essay['content']
    citation   = essay.get('citation_style', 'apa')
    runs_for_essay = []

    for run in range(1, RUNS_PER_ESSAY + 1):
        try:
            t0 = time.perf_counter()
            context = await supervisor.run_evaluation(
                content=essay_text,
                citation_style=citation,
            )
            wall_ms = (time.perf_counter() - t0) * 1000

            runs_for_essay.append({
                'overall_score':   context.overall_score,
                'worker_scores':   dict(context.final_scores),
                'critic_approved': context.critic_review.approved if context.critic_review else True,
                'wall_ms':         wall_ms,
            })
        except Exception as exc:
            failed.append((essay_id, run, str(exc)))
            print(f'  [FAIL] ID={essay_id} run={run}: {exc}')

    if runs_for_essay:
        records[essay_id] = runs_for_essay
        mean_score = st.mean(r['overall_score'] for r in runs_for_essay)
        gold_val   = gold_scores[essay_id]
        print(f'[{i+1}/{total_essays}] ID={essay_id:>4} | '
              f'gold={gold_val:>4.0f} | '
              f'pred_mean={mean_score:>6.2f} | '
              f'runs={len(runs_for_essay)}')

print(f'\nDone. {len(records)} essays evaluated, {len(failed)} run failures.')

## Cell 8 — Rescale Predictions to Gold Score Range (2–12)

Two rescaling strategies are computed, consistent with `compute_agreement.py`:

- **Standard rescaling**: linear map from [0, 100] to [2, 12] — applies the formula from Chapter 3.
- **Calibrated rescaling**: linear map from the *observed* prediction range to [2, 12] — corrects for
  systematic over- or under-scoring without changing the ordinal ranking.

In [ ]:
import math

def rescale_standard(score_100, lo=GOLD_MIN, hi=GOLD_MAX):
    """Chapter 3 formula: y = a + (S/100) * (b - a), rounded to nearest integer."""
    return int(round(lo + (score_100 / 100.0) * (hi - lo)))

# Build per-essay prediction vectors
essay_ids     = sorted(records.keys(), key=lambda x: int(x))
raw_means     = []   # mean raw 0-100 score per essay
raw_stdevs    = []   # stdev across runs (score stability)
gold_int      = []   # gold scores as integers
gold_float    = []   # gold scores as floats
pred_std      = []   # standard-rescaled predictions
worker_cols   = {}   # {worker_name: [mean score per essay]}
wall_times    = []   # all individual run wall times
critic_approvals = 0
total_runs_counted = 0

for eid in essay_ids:
    runs = records[eid]
    raw_scores = [r['overall_score'] for r in runs]
    mean_raw   = st.mean(raw_scores)
    stdev_raw  = st.stdev(raw_scores) if len(raw_scores) > 1 else 0.0

    raw_means.append(mean_raw)
    raw_stdevs.append(stdev_raw)
    gold_int.append(int(gold_scores[eid]))
    gold_float.append(float(gold_scores[eid]))
    pred_std.append(rescale_standard(mean_raw))

    for r in runs:
        wall_times.append(r['wall_ms'])
        if r['critic_approved']:
            critic_approvals += 1
        total_runs_counted += 1
        for w_name, w_score in r['worker_scores'].items():
            worker_cols.setdefault(w_name, []).append(float(w_score))

# Calibrated rescaling: map observed range → [2, 12]
obs_min   = min(raw_means)
obs_max   = max(raw_means)
obs_range = obs_max - obs_min if obs_max > obs_min else 1.0

pred_cal = [
    int(round(GOLD_MIN + ((s - obs_min) / obs_range) * (GOLD_MAX - GOLD_MIN)))
    for s in raw_means
]

print(f'Essays in analysis : {len(essay_ids)}')
print(f'Observed raw range : [{obs_min:.2f}, {obs_max:.2f}]')
print(f'Gold range         : [{min(gold_int)}, {max(gold_int)}]')
print(f'Standard pred range: [{min(pred_std)}, {max(pred_std)}]')
print(f'Calibrated pred rng: [{min(pred_cal)}, {max(pred_cal)}]')

## Cell 9 — Compute QWK, MAE, and Spearman Correlation

In [ ]:
# ── QWK ────────────────────────────────────────────────────────
def quadratic_weighted_kappa(y_true, y_pred, min_r=GOLD_MIN, max_r=GOLD_MAX):
    num_ratings = max_r - min_r + 1
    o = [[0] * num_ratings for _ in range(num_ratings)]
    for t, p in zip(y_true, y_pred):
        if min_r <= t <= max_r and min_r <= p <= max_r:
            o[t - min_r][p - min_r] += 1
    w = [[float((i - j) ** 2) / float((num_ratings - 1) ** 2)
          for j in range(num_ratings)] for i in range(num_ratings)]
    hist_t = [0] * num_ratings
    hist_p = [0] * num_ratings
    for t, p in zip(y_true, y_pred):
        if min_r <= t <= max_r and min_r <= p <= max_r:
            hist_t[t - min_r] += 1
            hist_p[p - min_r] += 1
    n = sum(hist_t)
    if n == 0:
        return 0.0
    e = [[float(hist_t[i] * hist_p[j]) / n
          for j in range(num_ratings)] for i in range(num_ratings)]
    num = sum(w[i][j] * o[i][j] for i in range(num_ratings) for j in range(num_ratings))
    den = sum(w[i][j] * e[i][j] for i in range(num_ratings) for j in range(num_ratings))
    return 1.0 if den == 0.0 else 1.0 - (num / den)

# ── MAE ────────────────────────────────────────────────────────
def mean_absolute_error(y_true, y_pred):
    return sum(abs(t - p) for t, p in zip(y_true, y_pred)) / len(y_true)

# ── Spearman ───────────────────────────────────────────────────
def spearman_correlation(x, y):
    """Rank-based Spearman correlation (no external libraries needed)."""
    n = len(x)
    if n < 2:
        return 0.0

    def rank(vals):
        sorted_idx = sorted(range(n), key=lambda i: vals[i])
        r = [0.0] * n
        i = 0
        while i < n:
            j = i
            while j < n - 1 and vals[sorted_idx[j]] == vals[sorted_idx[j + 1]]:
                j += 1
            avg_rank = (i + j) / 2.0 + 1  # 1-based average rank for ties
            for k in range(i, j + 1):
                r[sorted_idx[k]] = avg_rank
            i = j + 1
        return r

    rx, ry   = rank(x), rank(y)
    mean_rx  = sum(rx) / n
    mean_ry  = sum(ry) / n
    num      = sum((rx[i] - mean_rx) * (ry[i] - mean_ry) for i in range(n))
    den_x    = sum((rx[i] - mean_rx) ** 2 for i in range(n))
    den_y    = sum((ry[i] - mean_ry) ** 2 for i in range(n))
    if den_x == 0.0 or den_y == 0.0:
        return 0.0
    return num / math.sqrt(den_x * den_y)

# ── Compute ────────────────────────────────────────────────────
pred_std_float = [float(p) for p in pred_std]
pred_cal_float = [float(p) for p in pred_cal]

qwk_std  = quadratic_weighted_kappa(gold_int, pred_std)
mae_std  = mean_absolute_error(gold_float, pred_std_float)
spr_std  = spearman_correlation(gold_float, pred_std_float)

qwk_cal  = quadratic_weighted_kappa(gold_int, pred_cal)
mae_cal  = mean_absolute_error(gold_float, pred_cal_float)
spr_cal  = spearman_correlation(gold_float, pred_cal_float)

mean_stdev = st.mean(raw_stdevs)  # average score stability across runs

print('Standard rescaling   [y = 2 + (S/100) × 10]')
print(f'  QWK      : {qwk_std:.4f}')
print(f'  MAE      : {mae_std:.4f}')
print(f'  Spearman : {spr_std:.4f}')
print()
print('Calibrated rescaling [observed range → 2–12]')
print(f'  QWK      : {qwk_cal:.4f}')
print(f'  MAE      : {mae_cal:.4f}')
print(f'  Spearman : {spr_cal:.4f}')
print()
print(f'Score stability (mean stdev across runs): {mean_stdev:.4f}')

## Cell 10 — Operational Metrics (Latency, Critic, Worker Breakdown)

In [ ]:
def percentile(vals, p):
    xs = sorted(vals)
    if not xs:
        return 0.0
    k  = (len(xs) - 1) * p
    lo = math.floor(k)
    hi = math.ceil(k)
    return xs[lo] if lo == hi else xs[lo] * (hi - k) + xs[hi] * (k - lo)

p50  = percentile(wall_times, 0.50)
p95  = percentile(wall_times, 0.95)
mean_wall = st.mean(wall_times)
critic_rate = critic_approvals / total_runs_counted * 100

print(f'Wall latency  P50 : {p50:.0f} ms')
print(f'Wall latency  P95 : {p95:.0f} ms')
print(f'Mean wall latency : {mean_wall:.0f} ms')
print(f'Critic approval   : {critic_approvals}/{total_runs_counted} ({critic_rate:.1f}%)')
print()

# Worker-level mean scores and correlation with gold
print(f'{"Worker Agent":<32} | {"Mean Score":>10} | {"Spearman r":>10}')
print('-' * 58)

# Build per-essay worker means aligned to gold
for w_name in sorted(worker_cols.keys()):
    all_vals = worker_cols[w_name]
    # worker_cols holds one value per run, not per essay — collapse to essay-level means
    worker_essay_means = []
    for eid in essay_ids:
        runs = records[eid]
        vals_for_essay = [r['worker_scores'].get(w_name, 0.0) for r in runs]
        worker_essay_means.append(st.mean(vals_for_essay))

    mean_score = st.mean(worker_essay_means)
    corr       = spearman_correlation(gold_float, worker_essay_means) if len(worker_essay_means) > 1 else 0.0
    print(f'{w_name:<32} | {mean_score:>10.2f} | {corr:>10.4f}')

## Cell 11 — Full Summary

In [ ]:
print('=' * 62)
print('AWREAS SCORING BENCHMARK — SUMMARY')
print('=' * 62)
print(f'Corpus          : ASAP Prompt 1 (Shermis & Hamner, 2014)')
print(f'Gold score range: {GOLD_MIN}–{GOLD_MAX}')
print(f'Essays evaluated: {len(essay_ids)}')
print(f'Runs per essay  : {RUNS_PER_ESSAY}')
print(f'Provider / Model: {provider} / {model_name}')
print()
print('Standard Rescaling  [y = 2 + (S/100) × 10]')
print(f'  QWK              : {qwk_std:.4f}')
print(f'  MAE              : {mae_std:.4f}')
print(f'  Spearman rho     : {spr_std:.4f}')
print()
print('Calibrated Rescaling [observed range → 2–12]')
print(f'  QWK              : {qwk_cal:.4f}')
print(f'  MAE              : {mae_cal:.4f}')
print(f'  Spearman rho     : {spr_cal:.4f}')
print(f'  Observed raw rng : [{obs_min:.2f}, {obs_max:.2f}]')
print()
print(f'Score stability (mean stdev): {mean_stdev:.4f}')
print(f'Critic approval rate        : {critic_rate:.1f}%')
print(f'Wall latency P50 / P95      : {p50:.0f} ms / {p95:.0f} ms')
print('=' * 62)

## Cell 12 — Save Results to JSON

The output file is compatible with `scripts/compute_agreement.py` — you can
run that script on the saved file for additional analysis.

In [ ]:
from pathlib import Path

output_path = Path('data/asap/benchmark_results_prompt1.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

# Build results list in the same schema compute_agreement.py expects
results_list = []
for eid in essay_ids:
    for r in records[eid]:
        results_list.append({
            'system':         'AWREAS_Full',
            'case_id':        eid,
            'ok':             True,
            'overall_score':  r['overall_score'],
            'worker_scores':  r['worker_scores'],
            'critic_approved': r['critic_approved'],
            'wall_ms':        r['wall_ms'],
        })

# Also embed the benchmark summary for quick reference
output = {
    'meta': {
        'corpus':        'ASAP Prompt 1',
        'essays':        len(essay_ids),
        'runs_per_essay': RUNS_PER_ESSAY,
        'provider':      provider,
        'model':         model_name,
        'gold_range':    [GOLD_MIN, GOLD_MAX],
    },
    'aggregate': {
        'qwk_standard':    round(qwk_std, 4),
        'mae_standard':    round(mae_std, 4),
        'spearman_standard': round(spr_std, 4),
        'qwk_calibrated':  round(qwk_cal, 4),
        'mae_calibrated':  round(mae_cal, 4),
        'spearman_calibrated': round(spr_cal, 4),
        'mean_score_stdev': round(mean_stdev, 4),
        'critic_approval_pct': round(critic_rate, 2),
        'wall_p50_ms':     round(p50, 1),
        'wall_p95_ms':     round(p95, 1),
    },
    'results': results_list,
}

output_path.write_text(json.dumps(output, indent=2), encoding='utf-8')
print(f'Saved to {output_path}  ({len(results_list)} run records)')

## Cell 13 — Back Up to Google Drive

In [ ]:
import shutil

drive_dest = '/content/drive/MyDrive/AWREAS_Scoring_Results'
os.makedirs(drive_dest, exist_ok=True)
dest = shutil.copy(str(output_path), drive_dest)
print(f'Backed up to Google Drive: {dest}')